<div dir="rtl" align="right">

# كثافةُ الطاقةِ الطيفيّةِ \(PSD\)

**مجموعةُ البياناتِ**: PhysioNet Auditory EEG  
**القنواتُ**: P4, Cz, F8, T7  
**معدّلُ أخذِ العيناتِ**: 200 Hz  
**المُشاركُ**: 1

---

## نظرةٌ عامّةٌ

كثافةُ الطاقةِ الطيفيّةِ تَكشفُ توزيعَ طاقةِ الإشارةِ على الترددات. نَستخدمُ طريقةَ ويلش لِحسابِ \LR{PSD} للقنواتِ الأربعِ، ونُحلّلُ توزيعَ الطاقةِ على النطاقاتِ الدماغيّةِ الخمس.

## المُخرجاتُ المُتوقّعةُ

- مخططٌ طيفيٌّ للقنواتِ الأربعِ على مقياسٍ لوغاريتميّ
- مخططٌ شريطيٌّ لِتوزيعِ الطاقةِ على النطاقاتِ الخمسِ للقناةِ P4
- تَركّزُ الطاقةِ في النطاقاتِ المنخفضةِ وانخفاضُها في العالية

## المُعاملاتُ الأساسيةُ

| المُعاملُ | القيمةُ | المعنى |
| --- | --- | --- |
| nperseg | 1024 | طولُ الجزءِ |
| noverlap | 512 | التداخلُ بينَ الأجزاءِ |
| النطاقاتُ | 5 | دلتا، ثيتا، ألفا، بيتا، غاما |

</div>

<div dir="rtl" align="right">

## 1. تثبيتُ المكتباتِ

</div>

In [ ]:
!pip install scipy numpy plotly mne wfdb


<div dir="rtl" align="right">

## 2. استنساخُ المستودعِ وتنزيلُ بياناتِ مُشاركٍ واحدٍ

نَنزّلُ مُشاركًا واحدًا فقط (`--subjects 1`) لتسريعِ التجربةِ في بيئةِ Colab.

</div>

In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')


In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.dat')):
    !python data/download_local.py --output data/local --subjects 1


<div dir="rtl" align="right">

## 3. تحميلُ إشارةِ EEG

نحمّلُ تسجيلَ المُشاركِ 1 في التجربةِ 1، الجلسةِ 2.

</div>

In [ ]:
import numpy as np
from utils.eeg_loader import load_local_eeg

timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=1, experiment=1, session=2
)
fs = 200

print(f'Channels: {ch_names}')
print(f'Signal length: {len(eeg_data)} samples ({len(eeg_data)/fs:.1f} seconds)')


<div dir="rtl" align="right">

## 4. تطبيقُ تحليلِ \LR{PSD}

نَحسبُ كثافةَ الطاقةِ الطيفيّةَ بِطريقةِ ويلش لِجميعِ القنواتِ، ثمّ نُحلّلُ توزيعَ الطاقةِ على النطاقاتِ.

</div>

In [ ]:
from scipy.signal import welch

NPERSEG = 1024
NOVERLAP = 512
BANDS = [
    ('Delta', 0.5, 4, 'green'),
    ('Theta', 4, 8, 'blue'),
    ('Alpha', 8, 13, 'orange'),
    ('Beta', 13, 30, 'red'),
    ('Gamma', 30, 80, 'purple'),
]

psd_results = {}
for i, name in enumerate(ch_names):
    freqs, psd = welch(eeg_data[:, i], fs=fs, nperseg=NPERSEG, noverlap=NOVERLAP)
    psd_results[name] = (freqs, psd)

freqs_p4, psd_p4 = psd_results[ch_names[0]]
band_powers = []
for name, fmin, fmax, color in BANDS:
    mask = (freqs_p4 >= fmin) & (freqs_p4 <= fmax)
    band_powers.append(np.trapezoid(psd_p4[mask], freqs_p4[mask]))

print(f'Frequency resolution: {freqs_p4[1]-freqs_p4[0]:.2f} Hz')
print(f'Band powers (P4): {[f"{b[0]}={p:.1f}" for b,p in zip(BANDS, band_powers)]}')


<div dir="rtl" align="right">

## 5. رسمٌ تفاعليٌّ

**علامَ تُلاحظُ؟**

- الطاقةُ تَتركّزُ في النطاقاتِ المنخفضةِ وتَنخفضُ في العالية
- المقياسُ اللوغاريتميُّ يَكشفُ التفاصيلَ في جميعِ النطاقات
- النطاقاتُ المُظلّلةُ تُمثّلُ موجاتِ الدماغِ الخمسَ


</div>

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

colors = ['blue', 'orange', 'green', 'red']

fig = make_subplots(rows=2, cols=1, shared_xaxes=False,
                    subplot_titles=('PSD - All channels (Welch, log scale)',
                                    'Band power distribution - Channel P4'))
for i, name in enumerate(ch_names):
    f, p = psd_results[name]
    mask = f <= 80
    fig.add_trace(go.Scatter(x=f[mask], y=p[mask], name=name,
                             line=dict(color=colors[i], width=1)), row=1, col=1)
for bname, fmin, fmax, bcolor in BANDS:
    fig.add_vrect(x0=fmin, x1=fmax, fillcolor=bcolor, opacity=0.08,
                  line_width=0, row=1, col=1)
fig.update_xaxes(range=[0, 80], row=1, col=1)
fig.update_yaxes(type='log', row=1, col=1)

bar_colors = [b[3] for b in BANDS]
bar_names = [b[0] for b in BANDS]
fig.add_trace(go.Bar(x=bar_names, y=band_powers, marker_color=bar_colors,
                     name='Band power'), row=2, col=1)

fig.update_layout(height=800, title_text='Power Spectral Density Analysis',
                  xaxis_title='Frequency (Hz)', yaxis_title='PSD (uV^2/Hz)',
                  xaxis2_title='Frequency band', yaxis2_title='Absolute power (uV^2)',
                  showlegend=True)
fig.show()


<div dir="rtl" align="right">

## خلاصةٌ

- \LR{PSD} تَكشفُ توزيعَ طاقةِ الإشارةِ على الترددات
- طريقةُ ويلش تُقلّلُ التذبذبَ بِالتوسيطِ على أجزاءٍ مُتداخلة
- النطاقاتُ الدماغيّةُ الخمسُ تَظهرُ بِوضوحٍ في الطيفِ
- المقياسُ اللوغاريتميُّ ضروريٌّ لِكشفِ التفاصيلِ في النطاقاتِ العالية


</div>